# Hyperbolic Tree Builder - Test Notebook

Tests for `hyperbolic_tree.py`: projection to Lorentz manifold and parent-child tree construction.

In [8]:
import sys
sys.path.insert(0, '..')

import torch
from hhs.hyperbolic_tree import build_trees, project_to_manifold, build_parent_child_trees
from geoopt.manifolds.lorentz import Lorentz

## 1. Test projection to Lorentz manifold

In [9]:
manifold = Lorentz(k=1.0).double()
vectors = torch.randn(5, 4).double()

print("Input shape:", vectors.shape)
projected = project_to_manifold(vectors, manifold)
print("Projected shape:", projected.shape)
print("Projected vectors:")
print(projected)

Input shape: torch.Size([5, 4])
Projected shape: torch.Size([5, 5])
Projected vectors:
tensor([[ 1.3499,  0.6712,  0.3828,  0.4640, -0.0993],
        [ 7.6669,  0.2720,  4.1765, -0.8078,  6.2938],
        [ 2.4711, -0.1432, -0.9992, -0.3812, -1.9855],
        [ 7.8651,  0.1583, -4.0219,  6.6724,  0.3725],
        [ 1.9531,  0.6108, -0.8746,  0.9834, -0.8422]], dtype=torch.float64)


In [10]:
# Verify all points satisfy the Lorentz constraint: -x0^2 + x1^2 + ... + xd^2 = -k
ok, reason = manifold._check_point_on_manifold(projected)
print(f"Points on manifold: {ok}")
print(f"Reason: {reason}")
assert ok, f"Projection failed: {reason}"

Points on manifold: True
Reason: None


## 2. Test pairwise distances

In [11]:
parents = torch.randn(3, 4).double()
children = torch.randn(8, 4).double()

manifold_d = Lorentz(k=1.0).double()
parents_proj = project_to_manifold(parents, manifold_d)
children_proj = project_to_manifold(children, manifold_d)

dist_matrix = manifold_d.cdist(parents_proj, children_proj)
print("Distance matrix shape:", dist_matrix.shape)
print("Distance matrix:")
print(dist_matrix)
print()
print("All distances positive:", (dist_matrix >= 0).all().item())

Distance matrix shape: torch.Size([3, 8])
Distance matrix:
tensor([[4.8086, 3.7756, 3.5595, 3.8716, 4.0091, 4.6416, 3.5799, 2.5983],
        [3.4238, 3.7903, 1.7701, 4.0894, 2.7768, 4.0554, 2.1943, 4.4080],
        [3.2939, 3.2926, 1.6920, 3.5768, 2.4720, 3.7757, 0.6014, 3.5025]],
       dtype=torch.float64)

All distances positive: True


## 3. Test build_parent_child_trees

In [12]:
tree = build_parent_child_trees(parents_proj, children_proj, manifold_d)
print("Tree:")
for parent, children_set in tree.items():
    print(f"  {parent}: {children_set}")

total = sum(len(v) for v in tree.values())
print(f"\nTotal children assigned: {total}")
assert total == 8, f"Expected 8, got {total}"

Tree:
  parent_0: {'child_7'}
  parent_1: set()
  parent_2: {'child_2', 'child_0', 'child_1', 'child_4', 'child_5', 'child_6', 'child_3'}

Total children assigned: 8


## 4. Test full pipeline (build_trees)

In [13]:
parent_vecs = torch.randn(4, 6).double()
child_vecs = torch.randn(12, 6).double()

tree = build_trees(
    parent_vecs, child_vecs,
    curvature=1.0,
    parent_labels=["scene", "object_A", "object_B", "object_C"],
    child_labels=[f"slot_{i}" for i in range(12)],
)

print("Tree with custom labels:")
for parent, children_set in tree.items():
    print(f"  {parent}: {children_set}")

total = sum(len(v) for v in tree.values())
assert total == 12, f"Expected 12, got {total}"
assert len(tree) == 4, f"Expected 4 parents, got {len(tree)}"
print("\nAll assertions passed!")

Tree with custom labels:
  scene: {'slot_9', 'slot_6', 'slot_8', 'slot_10'}
  object_A: {'slot_2', 'slot_4', 'slot_5', 'slot_1', 'slot_11', 'slot_7', 'slot_3'}
  object_B: {'slot_0'}
  object_C: set()

All assertions passed!


## 5. Edge case: empty inputs

In [14]:
empty_tree = build_trees(torch.randn(3, 5).double(), torch.randn(0, 5).double())
print("Empty children:", empty_tree)
assert all(len(v) == 0 for v in empty_tree.values())
print("Edge case passed!")

Empty children: {'parent_0': set(), 'parent_1': set(), 'parent_2': set()}
Edge case passed!
